# Long-context methods: build the wall, then knock it down

A model trained on 4K tokens does not magically read 128K. Three walls stand in the way:
**positional generalization** (the model meets positions it never trained on), **attention
compute** ($O(N^2)$), and **KV-cache memory** (linear in $N$). This notebook attacks the
*positional* wall from scratch — the one you can fully see in a few lines of PyTorch — and
demonstrates the two structural tricks that make long context tractable.

We build and **prove** three things, each typed once in the companion script
[`long_context_methods.py`](long_context_methods.py) and mirrored here cell-for-cell:

1. **RoPE + Position Interpolation** — rotary embeddings are *relative*; naive extrapolation
   past the training length demands rotation angles the model never saw; **Position
   Interpolation** (Chen et al. 2023) squeezes the position index back into the trained range.
2. **Sliding-window receptive field** — one window layer is bounded, but stacking $L$ layers
   reaches $\approx L \times W$ tokens (the Mistral argument).
3. **StreamingLLM attention sinks** (Xiao et al. 2023) — a few leading tokens absorb the
   softmax's spare probability mass; evict them and the distribution collapses.

Every `assert` runs **before** any interpretation is printed. The trace is **pinned to CPU**
and labelled honestly so the printed numbers are reproducible on any host.

> **Step 1 — imports and named constants.** No magic numbers in the logic: the head
dimension, RoPE base, train/target lengths, window width, and StreamingLLM sink/window
sizes are all hoisted here. `DEVICE` auto-detects CUDA/MPS/CPU so the code runs anywhere,
but the numeric trace below is pinned to CPU on purpose.

In [1]:
from __future__ import annotations
import torch

HEAD_DIM = 8          # tiny even head dimension so RoPE's d/2 frequency pairs print by eye
ROPE_BASE = 10_000.0  # theta base from the RoPE paper; sets how fast each pair rotates
TRAIN_LEN = 16        # context length the toy model was "trained" on
TARGET_LEN = 64       # the longer context we want to extend to (4x training)
WINDOW = 4            # sliding-window width W: token attends to itself + (W-1) previous
N_LAYERS = 5          # depth used to demonstrate receptive-field growth
N_SINK = 4            # StreamingLLM: number of leading "sink" tokens to always keep
RECENT_W = 6          # StreamingLLM: size of the recent window kept alongside the sinks
SEED = 0
NEG_INF = float("-inf")  # masked scores -> -inf -> exactly 0 after softmax

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"device: cpu (trace pinned for reproducibility; DEVICE detected = {DEVICE})")
print("torch:", torch.__version__)

device: cpu (trace pinned for reproducibility; DEVICE detected = mps)
torch: 2.12.0


## Demo 1 — RoPE and Position Interpolation

**Rotary Position Embedding (RoPE)** rotates each adjacent dimension *pair* of a query/key
vector by an angle proportional to its position. Because both $q$ and $k$ are rotated, their
dot product depends only on the *difference* of their positions — RoPE is **relative**. The
catch for long context: each frequency pair's angle grows with position, so a position far
beyond training produces an angle the model **never saw**, and attention degrades. **Position
Interpolation** (Chen et al. 2023) fixes this by scaling every position index by
$L_\text{train}/L_\text{target}$ before computing angles, keeping them inside the trained range.

> **Step 2 — RoPE angles.** $\theta_{m,i} = m \cdot \text{base}^{-2i/d}$: position $m$ times the
angular frequency of pair $i$. Low-$i$ pairs rotate fast (short wavelength), high-$i$ pairs
slowly — like a bank of clocks ticking at different rates. `arange` is built on `DEVICE`.

In [2]:
def rope_angles(positions, head_dim=HEAD_DIM, base=ROPE_BASE, *, device=DEVICE):
    """theta_{m,i} = m * base^(-2i/d). positions:(T,) -> (T, head_dim/2) angles."""
    half = head_dim // 2
    inv_freq = base ** (-torch.arange(0, half, device=device, dtype=torch.float32) * 2.0 / head_dim)
    return positions[:, None].to(torch.float32) * inv_freq[None, :]   # outer product (T, half)

> **Step 3 — apply the rotation.** For a pair $(x_\text{even}, x_\text{odd})$ at angle $t$ this is the
2-D rotation $[\cos t, -\sin t;\ \sin t, \cos t]$. Rotating both $q$ and $k$ and dotting them
leaves a function of the relative offset $(i-j)$ — that is the whole RoPE trick.

In [3]:
def apply_rope(x, angles):
    """Rotate each adjacent dim pair of x:(T,head_dim) by angles:(T,head_dim/2)."""
    cos, sin = torch.cos(angles), torch.sin(angles)
    x_even, x_odd = x[:, 0::2], x[:, 1::2]
    rot_even = x_even * cos - x_odd * sin        # first row of the 2-D rotation matrix
    rot_odd  = x_even * sin + x_odd * cos         # second row
    out = torch.empty_like(x)                     # allocate on x's device/dtype
    out[:, 0::2] = rot_even
    out[:, 1::2] = rot_odd
    return out

> **Step 4 — a scored attention logit.** Rotate $q$ at `pos_q` and $k$ at `pos_k`, then dot
them. The `scale` argument multiplies each raw position *before* angles: `scale=1` is
standard RoPE; `scale = TRAIN_LEN/TARGET_LEN` is **Position Interpolation**.

In [4]:
def rope_score(q, k, pos_q, pos_k, *, scale, device="cpu"):
    pq = torch.tensor([pos_q * scale], device=device)
    pk = torch.tensor([pos_k * scale], device=device)
    q_rot = apply_rope(q[None, :], rope_angles(pq, device=device))[0]
    k_rot = apply_rope(k[None, :], rope_angles(pk, device=device))[0]
    return float((q_rot @ k_rot).item())

> **Step 5a — RoPE is relative.** Same offset, different absolute positions → **same score**.
We compute the logit for positions (2,5) and (10,13) — both offset $-3$ — and assert they
match. This is the property that lets RoPE generalize at all.

In [5]:
torch.manual_seed(SEED)
cpu = "cpu"            # pin the trace to CPU so these numbers are reproducible
q = torch.randn(HEAD_DIM, device=cpu)
k = torch.randn(HEAD_DIM, device=cpu)

s_2_5   = rope_score(q, k, 2, 5,   scale=1.0, device=cpu)   # offset -3
s_10_13 = rope_score(q, k, 10, 13, scale=1.0, device=cpu)   # offset -3, far away
assert abs(s_2_5 - s_10_13) < 1e-4, "RoPE score must depend only on (i - j)"
print(f"score(pos 2 vs 5)   = {s_2_5: .6f}")
print(f"score(pos 10 vs 13) = {s_10_13: .6f}  (same offset -3)")
print(f"max abs diff = {abs(s_2_5 - s_10_13):.2e}  -> relative property holds")

score(pos 2 vs 5)   =  1.585057
score(pos 10 vs 13) =  1.585057  (same offset -3)
max abs diff = 0.00e+00  -> relative property holds


> **Step 5b — the wall, and the fix.** The fastest-rotating pair ($i=0$) wraps soonest, so
its angle at the largest position is what overshoots. Naive extrapolation to position 63
demands an angle ~4x past anything seen in training; **PI** scales the index by
$16/64=0.25$, dropping the largest angle back to ~15.75 rad — essentially the trained
ceiling. The asserts encode exactly that: naive overshoots, PI lands back in-range.

In [6]:
trained_max = rope_angles(torch.tensor([TRAIN_LEN - 1.0], device=cpu), device=cpu)[0, 0]
naive_max   = rope_angles(torch.tensor([TARGET_LEN - 1.0], device=cpu), device=cpu)[0, 0]
pi_scale    = TRAIN_LEN / TARGET_LEN                       # Position Interpolation squeeze
pi_max      = rope_angles(torch.tensor([(TARGET_LEN - 1.0) * pi_scale], device=cpu), device=cpu)[0, 0]

assert naive_max > trained_max * 2,            "naive extension should overshoot"
assert float(pi_max) <= float(trained_max) + 1.0, "PI should land at ~trained max angle"
assert float(pi_max) < float(naive_max) / 3,   "PI must be far below the naive angle"
print(f"trained max angle (pos {TRAIN_LEN-1})  = {float(trained_max):.4f} rad")
print(f"naive   max angle (pos {TARGET_LEN-1})  = {float(naive_max):.4f} rad  "
      f"-> {float(naive_max/trained_max):.1f}x past trained")
print(f"PI      max angle (scale {pi_scale:.3f}) = {float(pi_max):.4f} rad  -> back inside trained range")

trained max angle (pos 15)  = 15.0000 rad
naive   max angle (pos 63)  = 63.0000 rad  -> 4.2x past trained
PI      max angle (scale 0.250) = 15.7500 rad  -> back inside trained range


## Demo 2 — sliding-window receptive field

**Sliding-window attention** (Mistral) caps the KV cache: each token attends only to the last
$W$ tokens, so the cache never exceeds $W$ entries. The worry is that a token can then only
*see* $W$ tokens back. Not so — stacking layers grows the reach, exactly like stacking
convolutions grows a CNN's receptive field. Over $L$ layers, information propagates
window-by-window up the stack to an effective reach of $\approx L \times W$ tokens.

> **Step 6 — the window mask.** `True` where query $i$ may attend to key $j$: causal AND
within the window, i.e. $0 \le i-j < W$. Everything older than the window or in the future
is masked.

In [7]:
def sliding_window_mask(seq_len, window=WINDOW, *, device=DEVICE):
    idx = torch.arange(seq_len, device=device)
    offset = idx[:, None] - idx[None, :]              # offset[i, j] = i - j
    return (offset >= 0) & (offset < window)          # not future, and within the window

> **Step 7 — one layer is bounded.** Build a 12-token mask, run one masked attention row,
and assert the last token attends to **exactly** keys $[8,9,10,11]$ with ~zero weight on
anything older. A token literally cannot see beyond its window in a single layer.

In [8]:
torch.manual_seed(SEED)
seq_len = 12
mask = sliding_window_mask(seq_len, device=cpu)
w_qk = torch.randn(2, 2, device=cpu) * 0.1
x = torch.randn(seq_len, 2, device=cpu) * 0.1
proj = x @ w_qk
scores = (proj @ proj.transpose(-1, -2) * (2 ** -0.5)).masked_fill(~mask, NEG_INF)
weights = torch.softmax(scores, dim=-1)

last = seq_len - 1
allowed = mask[last].nonzero().flatten().tolist()
forbidden_weight = weights[last, : last - WINDOW + 1].sum().item()
assert allowed == list(range(last - WINDOW + 1, last + 1)), "window must be exactly the last W keys"
assert forbidden_weight < 1e-6, "out-of-window keys must receive ~0 weight"
print(f"window W = {WINDOW}; last token (pos {last}) may attend to keys {allowed}")
print(f"total attention weight on out-of-window keys = {forbidden_weight:.2e}")

window W = 4; last token (pos 11) may attend to keys [8, 9, 10, 11]
total attention weight on out-of-window keys = 0.00e+00


> **Step 8 — depth grows the reach.** `reachable_set(L, query)` expands the influence set by
one window per layer. We print the reach after 1..5 layers and assert the closed form
$1 + L\,(W-1)$ — the algebra behind "a $W{=}4096$ window over 32 layers reaches ~131K tokens."

In [9]:
def reachable_set(layer, query, window=WINDOW):
    reach = {query}
    for _ in range(layer):
        expanded = set()
        for pos in reach:
            lo = max(0, pos - (window - 1))           # window reaches window-1 tokens back
            expanded.update(range(lo, pos + 1))
        reach = expanded
    return reach

query = seq_len - 1
for layer in range(1, N_LAYERS + 1):
    reach = reachable_set(layer, query)
    span = query - min(reach) + 1
    print(f"after {layer} layer(s): reaches back {span:>2} tokens  (min pos {min(reach)})")

span = query - min(reachable_set(N_LAYERS, query)) + 1
expected = min(query + 1, 1 + N_LAYERS * (WINDOW - 1))
assert span == expected, f"reach {span} should equal 1 + L*(W-1) = {expected}"
print(f"\nassert: {N_LAYERS}-layer reach == 1 + L*(W-1) = {expected} tokens  -> depth multiplies the window")

after 1 layer(s): reaches back  4 tokens  (min pos 8)
after 2 layer(s): reaches back  7 tokens  (min pos 5)
after 3 layer(s): reaches back 10 tokens  (min pos 2)
after 4 layer(s): reaches back 12 tokens  (min pos 0)
after 5 layer(s): reaches back 12 tokens  (min pos 0)

assert: 5-layer reach == 1 + L*(W-1) = 12 tokens  -> depth multiplies the window


## Demo 3 — StreamingLLM attention sinks

For truly endless streams, keep only the recent window — but **softmax forces the attention
weights to sum to 1**, with no "attend to nothing" option. When a query matches no recent
token well, that mandatory mass pools on the first few tokens, visible from *every* position.
Evict those **sink** tokens and you delete a big chunk of the softmax **denominator**; the
survivors renormalize wildly. **StreamingLLM** keeps ~4 sinks + the recent window → stable,
infinite-length streaming at bounded memory.

> **Step 9 — set up the sink pattern.** We emulate the trained sink effect: the first few
keys align with *every* query (the `+ q*1.2` nudge), so they soak up spare mass. We then
build the full softmax and measure how much probability lands on the sinks.

In [10]:
torch.manual_seed(SEED)
seq_len = 40
scale = HEAD_DIM ** -0.5
q = torch.randn(HEAD_DIM, device=cpu)
keys = torch.randn(seq_len, HEAD_DIM, device=cpu)
keys[:N_SINK] += q * 1.2                          # sinks align with every query -> soak up mass
logits = (keys @ q) * scale

full = torch.softmax(logits, dim=-1)
sink_mass = full[:N_SINK].sum().item()
print(f"probability mass on the {N_SINK} sink tokens = {sink_mass:.3f}")

probability mass on the 4 sink tokens = 0.853


> **Step 10 — evict vs keep.** *Eviction* drops the sinks and renormalizes over the recent
window; *StreamingLLM* keeps sinks **and** the recent window. We compare each recent token's
weight to its weight in the full distribution and assert that keeping the sinks keeps the
recent weights far more stable — the mechanical reason streaming works.

In [11]:
def streaming_keep_mask(seq_len, n_sink=N_SINK, recent_w=RECENT_W, *, device=DEVICE):
    keep = torch.zeros(seq_len, dtype=torch.bool, device=device)
    keep[:n_sink] = True                          # the load-bearing sink tokens
    keep[-recent_w:] = True                       # the recent window
    return keep

# Eviction: keep only the recent window. Masking every index before seq_len - RECENT_W
# already drops the first N_SINK sink tokens too -- no separate evict[:N_SINK] needed.
evict = logits.clone()
evict[: seq_len - RECENT_W] = NEG_INF
w_evicted = torch.softmax(evict, dim=-1)

# StreamingLLM: keep sinks AND the recent window.
keep = streaming_keep_mask(seq_len, device=cpu)
w_stream = torch.softmax(logits.masked_fill(~keep, NEG_INF), dim=-1)

recent_idx = torch.arange(seq_len - RECENT_W, seq_len, device=cpu)
drift_evicted = (w_evicted[recent_idx] - full[recent_idx]).abs().max().item()
drift_stream  = (w_stream[recent_idx]  - full[recent_idx]).abs().max().item()
assert sink_mass > 0.3,               "sinks should absorb a large share of the mass"
assert drift_stream < drift_evicted,  "keeping sinks must keep recent weights more stable"
print(f"recent-window weight drift WITHOUT sinks = {drift_evicted:.3f}  (renormalized wildly)")
print(f"recent-window weight drift WITH    sinks = {drift_stream:.3f}  (stable)")
print("\nall asserts passed.")

recent-window weight drift WITHOUT sinks = 0.295  (renormalized wildly)
recent-window weight drift WITH    sinks = 0.001  (stable)

all asserts passed.


## What we saw

- **RoPE is relative** — equal offsets gave bit-for-bit equal scores, the property that lets
  it generalize at all.
- **The positional wall is an *angle* problem** — naive extrapolation to 4x the training
  length demanded rotation angles ~4x past anything seen; **Position Interpolation** squeezed
  them back inside the trained range. (NTK-aware / YaRN refine *which* frequencies to scale.)
- **A window is not a ceiling** — one layer is bounded to $W$, but $L$ layers reach
  $1 + L\,(W-1) \approx L\,W$ tokens, so cache-bounded sliding-window models still see far.
- **Attention sinks are load-bearing** — a few leading tokens absorb the softmax's spare
  mass; keep them and streaming is stable, evict them and the recent weights drift wildly.

These are the *positional* and *structural* halves of long context. The **compute** half
(FlashAttention/sparse attention) and the **memory** half (KV-cache compression, MLA,
quantization) are covered in
[Efficient Attention (FlashAttention)](../../06-Efficient-Attention-FlashAttention/06-Efficient-Attention-FlashAttention.md)
and [KV Cache](../../05-KV-Cache/05-KV-Cache.md).